# 🪡 31 · Picking particles with almost no labels: PU learning, masked autoencoders and meta-learning
### When ten clicks must find two thousand ribosomes — and why "unlabelled" is not "negative"

**BiologyToResearch** · Part G — Frontiers, biology, research practice · Module 31 of 34

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/cryoem/tutorial/BiologyToResearch/notebooks/31_picking_with_sparse_labels.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1–2 min); no GPU needed |
| **Core frameworks** | Python 3, PyTorch, NumPy, SciPy, Matplotlib |
| **Prerequisites** | Modules 04 (precision/recall), 05 (PyTorch), 11 (picking, NMS), 19 (few labels, self-supervision) |
| **Estimated time** | 150 min (reading + running + the ✏️ exercises) |
| **Lab papers covered** | **Uddin *et al.* 2025** (Brief. Bioinform. — TomoPicker, sparse-label localisation); **Xu, Zhan & Xu 2025** (WACV — CryoMAE); **Hosseini *et al.*** (Nature Methods — RobPicker, ⚠️ content not verified); Cai *et al.* 2019 (NeurIPS workshop — deep RL localisation) — see [`PAPER_ATLAS.md`](../PAPER_ATLAS.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis. All
> experiments are small 2D toys. They illustrate the core ideas of the papers; they do **not** reproduce their
> networks or numbers. Section 4 is a generic illustration of meta-learning, **not** RobPicker's algorithm (whose text
> we could not access). **All molecules and data are generated by code** (nothing is downloaded): realistic stand-ins
> with the size and symmetry of the PDB entries they are named after, and a β-galactosidase-sized map at 3.54 Å/voxel.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Explain** why treating unlabelled voxels as negatives biases a particle picker, and distinguish a classifier's
   *ranking* from its *calibration*.
2. **Compute** the class prior π from an expected particle count and particle size, as TomoPicker does.
3. **Implement** the non-negative PU risk (nnPU) and the generalized-expectation KL criterion (GE-KL), and **compare**
   them with naive positive–negative training.
4. **Build** a miniature masked autoencoder, add CryoMAE's self–cross similarity loss, and pick particles in a micrograph
   from 15 exemplars with an elbow threshold.
5. **Describe** what meta-learning aims at, using a Reptile toy that adapts a picker to new imaging conditions.
6. **Place** the lab's pickers on a ladder of supervision, from template-free to fully supervised.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit, molecules | 5 min |
| 1 · The sparse-label problem | a crowded slab, 10 labelled ribosomes | 10 min |
| 2 · Positive–unlabelled learning | PN vs nnPU vs GE-KL; ranking vs calibration | 40 min |
| 3 · CryoMAE-lite | masked autoencoder + exemplars + elbow | 35 min |
| 4 · Meta-learning (illustration) | Reptile: learning to adapt quickly | 25 min |
| 5 · The supervision ladder | where each lab picker sits | 10 min |
| 📄 / ✅ / 🧠 / 🚀 | paper connections, checkpoint, discussion, going further | 15 min |

## 🧫 The biological question

A *Schizosaccharomyces pombe* tomogram may contain a couple of thousand 80S ribosomes. Ribosome positions tell us where
proteins are being made — free in the cytosol, on the ER membrane, in polysomes — but picking them reliably usually
needs a trained network, and training normally needs hundreds of hand-picked examples per tomogram type. An expert can
afford to click maybe ten. The trouble is subtle: every ribosome the expert did **not** click is still a ribosome. A
method that learns "clicked = ribosome, everything else = not ribosome" is being taught a lie. This module is about
learning honestly from **positive and unlabelled** data — and about other ways to need fewer labels.

---
## 0 · Environment setup

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — fine, everything runs on a CPU.")
try:
    import torch
    print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
except ImportError:
    print("PyTorch not installed yet — the next cell installs it.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("matplotlib", "matplotlib"), ("torch", "torch")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import copy
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import ndimage

SEED = 0
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "image.cmap": "gray"})
print("NumPy", np.__version__, "| PyTorch", torch.__version__, "| device", DEVICE, "| seed", SEED)

### 🧰 Course toolkit

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import struct

import numpy as np
from scipy import ndimage

ZOO = {
    "4v4r": dict(name="70S ribosome", organism="bacterial (E. coli / T. thermophilus)", mass_kDa=2330, symmetry="C1"),
    "4v7r": dict(name="80S ribosome", organism="S. cerevisiae", mass_kDa=3330, symmetry="C1"),
    "5mrc": dict(name="mitochondrial ribosome", organism="S. cerevisiae", mass_kDa=3320, symmetry="C1"),
    "3j9i": dict(name="20S proteasome", organism="Thermoplasma acidophilum", mass_kDa=710, symmetry="D7"),
    "6utj": dict(name="singly capped proteasome", organism="Thermoplasma acidophilum", mass_kDa=910, symmetry="C7"),
    "4cr2": dict(name="26S proteasome", organism="S. cerevisiae", mass_kDa=1280, symmetry="C1"),
    "4v94": dict(name="TRiC/CCT chaperonin (×2)", organism="S. cerevisiae", mass_kDa=1910, symmetry="C2"),
    "1bxn": dict(name="RuBisCO (L4S4 half)", organism="Cupriavidus necator", mass_kDa=290, symmetry="C4"),
    "1qvr": dict(name="ClpB (3 of 6 subunits)", organism="Thermus thermophilus", mass_kDa=290, symmetry="C1"),
    "3cf3": dict(name="p97/VCP ATPase", organism="Mus musculus", mass_kDa=570, symmetry="C6"),
    "2cg9": dict(name="Hsp90–Sba1 chaperone", organism="S. cerevisiae", mass_kDa=190, symmetry="C2"),
    "1u6g": dict(name="Cand1–Cul1–Roc1 ligase", organism="H. sapiens", mass_kDa=260, symmetry="C1"),
    "3d2f": dict(name="Sse1p–Hsp70 chaperone", organism="S. cerevisiae / H. sapiens", mass_kDa=230, symmetry="C1"),
    "1s3x": dict(name="Hsp70 ATPase domain", organism="H. sapiens", mass_kDa=44, symmetry="C1"),
}
_BOND = dict(N_CA=1.458, CA_C=1.525, C_N=1.329, C_O=1.231)
_ANGLE = dict(N_CA_C=111.2, CA_C_N=116.2, C_N_CA=121.7, CA_C_O=120.5)
_SIDECHAIN = {
    "GLY": [], "ALA": [], "SER": [("OG", "O")], "CYS": [("SG", "S")], "VAL": [("CG1", "C"), ("CG2", "C")],
    "THR": [("OG1", "O"), ("CG2", "C")], "PRO": [("CG", "C"), ("CD", "C")],
    "LEU": [("CG", "C"), ("CD1", "C"), ("CD2", "C")], "ILE": [("CG1", "C"), ("CG2", "C"), ("CD1", "C")],
    "ASP": [("CG", "C"), ("OD1", "O"), ("OD2", "O")], "ASN": [("CG", "C"), ("OD1", "O"), ("ND2", "N")],
    "GLU": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("OE2", "O")], "GLN": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("NE2", "N")],
    "LYS": [("CG", "C"), ("CD", "C"), ("CE", "C"), ("NZ", "N")], "MET": [("CG", "C"), ("SD", "S"), ("CE", "C")],
    "ARG": [("CG", "C"), ("CD", "C"), ("NE", "N"), ("CZ", "C"), ("NH1", "N"), ("NH2", "N")],
    "HIS": [("CG", "C"), ("ND1", "N"), ("CD2", "C"), ("CE1", "C"), ("NE2", "N")],
    "PHE": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C")],
    "TYR": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C"), ("OH", "O")],
    "TRP": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("NE1", "N"), ("CE2", "C"), ("CE3", "C"), ("CZ2", "C"),
            ("CZ3", "C"), ("CH2", "C")],
}
_AA_FREQ = {"LEU": 9.9, "ALA": 8.3, "GLY": 7.1, "VAL": 6.9, "GLU": 6.8, "SER": 6.6, "ILE": 5.9, "LYS": 5.8, "ARG": 5.5,
            "ASP": 5.5, "THR": 5.3, "PRO": 4.7, "ASN": 4.1, "GLN": 3.9, "PHE": 3.9, "TYR": 2.9, "MET": 2.4, "HIS": 2.3,
            "CYS": 1.4, "TRP": 1.1}
def _nerf(a, b, c, bond, angle, torsion):
    """Place atom d so that |cd| = bond, angle(b,c,d) = angle and dihedral(a,b,c,d) = torsion (degrees)."""
    bc = (c - b) / np.linalg.norm(c - b)
    n = np.cross(b - a, bc)
    n /= np.linalg.norm(n)
    m = np.cross(n, bc)
    ang, tor = np.radians(angle), np.radians(torsion)
    d = np.array([-bond * np.cos(ang), bond * np.sin(ang) * np.cos(tor), bond * np.sin(ang) * np.sin(tor)])
    return c + d[0] * bc + d[1] * m + d[2] * n


def build_backbone(phi, psi, omega=None) -> np.ndarray:
    """Backbone atoms N, CA, C, O, CB (array n×5×3, Å) of a chain with the given torsion angles (degrees)."""
    n = len(phi)
    omega = np.full(n, 180.0) if omega is None else np.asarray(omega, float)
    N = [np.zeros(3)]
    CA = [np.array([_BOND["N_CA"], 0.0, 0.0])]
    C = [_nerf(np.array([0.0, 1.0, 0.0]), N[0], CA[0], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[0])]
    for i in range(n - 1):
        N.append(_nerf(N[i], CA[i], C[i], _BOND["C_N"], _ANGLE["CA_C_N"], psi[i]))
        CA.append(_nerf(CA[i], C[i], N[i + 1], _BOND["N_CA"], _ANGLE["C_N_CA"], omega[i]))
        C.append(_nerf(C[i], N[i + 1], CA[i + 1], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[i + 1]))
    N, CA, C = np.array(N), np.array(CA), np.array(C)
    O = np.array([_nerf(N[i], CA[i], C[i], _BOND["C_O"], _ANGLE["CA_C_O"], psi[i] + 180.0) for i in range(n)])
    b, c = CA - N, C - CA
    a = np.cross(b, c)
    CB = -0.58273431 * a + 0.56802827 * b - 0.54067466 * c + CA      # ideal Cβ position
    return np.stack([N, CA, C, O, CB], 1)


def _ideal_helix_template(n=160):
    """An ideal α-helix (φ = −57.8°, ψ = −47.0°) with its axis along +z through x = y = 0."""
    bb = build_backbone(np.full(n, -57.8), np.full(n, -47.0))
    ca = bb[:, 1]
    centre = ca.mean(0)
    u = np.linalg.svd(ca - centre)[2][0]
    if np.dot(ca[-1] - ca[0], u) < 0:
        u = -u
    # rotation taking u to z
    v = np.cross(u, [0, 0, 1.0]); s = np.linalg.norm(v); cth = u[2]
    K = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    Rm = np.eye(3) + K + K @ K * ((1 - cth) / (s ** 2 + 1e-12))
    xyz = (bb - centre) @ Rm.T
    axis_xy = xyz[:, 1, :2].mean(0)
    xyz[..., :2] -= axis_xy
    return xyz


_HELIX = _ideal_helix_template()
def _sidechains(bb, resnames, rng):
    """Approximate side-chain heavy atoms grown outward from Cβ (positions are schematic; counts/elements are right)."""
    names, elems, xyz, resid = [], [], [], []
    ca, cb = bb[:, 1], bb[:, 4]
    out = cb - ca
    out /= np.linalg.norm(out, axis=1, keepdims=True)
    for i, rn in enumerate(resnames):
        sc = _SIDECHAIN[rn]
        if not sc:
            continue
        perp = np.cross(out[i], rng.normal(size=3)); perp /= np.linalg.norm(perp)
        for k, (an, el) in enumerate(sc):
            depth = 1.0 + (k // 2)                               # schematic zig-zag outward
            pos = cb[i] + 1.5 * depth * out[i] + 0.9 * (1 if k % 2 else -1) * perp
            names.append(an); elems.append(el); xyz.append(pos); resid.append(i)
    return names, elems, np.array(xyz).reshape(-1, 3), np.array(resid, int)


def _loop_path(a, b, outward, step=3.8):
    """Cα positions of a loop from Cα `a` to Cα `b`: a circular arc bulging along `outward`, cut into equal chords so
    that every Cα–Cα distance (including the two ends) is exactly `step` Å."""
    c = np.linalg.norm(b - a)
    e = (b - a) / c
    u = outward - (outward @ e) * e                                  # bulge direction ⟂ to the chord
    if np.linalg.norm(u) < 1e-6:
        u = np.cross(e, [0.0, 0.0, 1.0]) if abs(e[2]) < 0.9 else np.cross(e, [1.0, 0.0, 0.0])
    u = u / np.linalg.norm(u)
    m = max(int(np.ceil(c / step)), 1)                               # m new residues → n = m + 1 equal chords
    n = m + 1
    lo, hi = 1e-9, 2 * np.pi - 1e-9                                  # arc angle Θ: sin(Θ/2) / sin(Θ/2n) = c / step
    for _ in range(60):
        th = (lo + hi) / 2
        lo, hi = (th, hi) if np.sin(th / 2) / np.sin(th / (2 * n)) > c / step else (lo, th)
    R = step / (2 * np.sin(th / (2 * n)))
    centre = (a + b) / 2 - R * np.cos(th / 2) * u
    beta = np.pi / 2 + th / 2 - th * np.arange(1, n) / n             # angles of the new points, a → b over the bulge
    return centre + R * (np.outer(np.cos(beta), e) + np.outer(np.sin(beta), u))


def _assemble_chain(ca_path_segments, rng, chain="A", start_resnum=1):
    """Turn helices (n×5×3 arrays) joined by loops into one chain with residue names and side chains."""
    blocks = []
    for k, seg in enumerate(ca_path_segments):
        blocks.append(seg)
        if k + 1 < len(ca_path_segments):
            a, b = seg[-1, 1], ca_path_segments[k + 1][0, 1]
            path = _loop_path(a, b, (a + b) / 2 - seg.reshape(-1, 3).mean(0))
            m = len(path)
            tang = np.gradient(path, axis=0) if m > 1 else np.array([b - a])
            tang /= np.linalg.norm(tang, axis=1, keepdims=True)
            side = np.cross(tang, rng.normal(size=3)); side /= np.linalg.norm(side, axis=1, keepdims=True)
            loop = np.stack([path - 1.2 * tang, path, path + 1.2 * tang, path + 1.2 * tang + 1.0 * side,
                             path - 1.3 * side], 1)
            blocks.append(loop)
    bb = np.concatenate(blocks, 0)
    aa = list(_AA_FREQ)
    p = np.array([_AA_FREQ[a] for a in aa]); p /= p.sum()
    resnames = rng.choice(aa, len(bb), p=p)
    bb_names = ["N", "CA", "C", "O", "CB"]
    names, elems, xyz, resid = [], [], [], []
    for i in range(len(bb)):
        for j, an in enumerate(bb_names):
            if an == "CB" and resnames[i] == "GLY":
                continue
            names.append(an); elems.append(an[0]); xyz.append(bb[i, j]); resid.append(i)
    sn, se, sx, sr = _sidechains(bb, resnames, rng)
    names += sn; elems += se; resid = np.concatenate([np.array(resid, int), sr])
    xyz = np.concatenate([np.array(xyz), sx], 0)
    order = np.argsort(resid, kind="stable")
    return dict(xyz=xyz[order].astype(np.float32), name=np.array(names)[order], element=np.array(elems)[order],
                resname=resnames[resid[order]], resnum=(resid[order] + start_resnum).astype(int),
                chain=np.full(len(order), chain))


def helix_bundle(radii_A, rng, chain="A", spacing=10.0):
    """A globular 'domain' that fills an ellipsoid (semi-axes in Å, long axis = local z) with antiparallel helices."""
    a, b, c = radii_A[0] + 2.5, radii_A[1] + 2.5, radii_A[2] + 1.5   # helices stop short of the surface: pad
    rows = []
    j = 0
    while (j * spacing * np.sqrt(3) / 2) <= b:
        rows.append(j); rows.append(-j) if j else None; j += 1
    pts = []
    for jj in sorted(set(rows)):
        y = jj * spacing * np.sqrt(3) / 2
        xs = np.arange(-a, a + 1e-6, spacing) + (spacing / 2 if jj % 2 else 0)
        row = [(x, y) for x in xs if (x / a) ** 2 + (y / b) ** 2 < 1]
        pts.append(row if (jj % 2 == 0) else row[::-1])
    pts = [p for row in pts for p in row]
    segs = []
    for k, (x, y) in enumerate(pts):
        h = c * np.sqrt(max(0.0, 1 - (x / a) ** 2 - (y / b) ** 2)) - 2.0
        n = int(2 * h / 1.5)
        if n < 6:
            continue
        seg = _HELIX[:n].copy()
        seg[..., 2] -= seg[:, 1, 2].mean()
        phase = rng.uniform(0, 2 * np.pi)
        Rz = np.array([[np.cos(phase), -np.sin(phase), 0], [np.sin(phase), np.cos(phase), 0], [0, 0, 1]])
        seg = seg @ Rz.T
        if len(segs) % 2 == 1:                                   # antiparallel neighbours: 180° turn about x
            seg = seg * np.array([1.0, -1.0, -1.0])
        seg = seg + np.array([x, y, rng.uniform(-1.0, 1.0)])
        segs.append(seg)
    return _assemble_chain(segs, rng, chain)


def _transform(atoms, R=np.eye(3), t=(0, 0, 0)):
    out = dict(atoms)
    out["xyz"] = (atoms["xyz"] @ np.asarray(R, float).T + np.asarray(t, float)).astype(np.float32)
    return out


def merge_atoms(parts):
    keys = parts[0].keys()
    return {k: np.concatenate([p[k] for p in parts]) for k in keys}


def _rz(a):
    return np.array([[np.cos(a), -np.sin(a), 0], [np.sin(a), np.cos(a), 0], [0, 0, 1.0]])


def _rx(a):
    return np.array([[1, 0, 0], [0, np.cos(a), -np.sin(a)], [0, np.sin(a), np.cos(a)]])


def _ry(a):
    return np.array([[np.cos(a), 0, np.sin(a)], [0, 1, 0], [-np.sin(a), 0, np.cos(a)]])


_CHAIN_IDS = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789"
def _ring(n, radius, z, radii, phase=0.0, tilt=None):
    """n identical subunits (semi-axes `radii` nm: radial, tangential, axial) around the z axis."""
    out = []
    for k in range(n):
        a = phase + 2 * np.pi * k / n
        R = _rz(a) if tilt is None else _rz(a) @ tilt
        out.append(dict(radii=radii, centre=(radius * np.cos(a), radius * np.sin(a), z), R=R, key=("ring", radii, tilt is None)))
    return out


def _lobe(radii, centre, R=None):
    return [dict(radii=radii, centre=centre, R=np.eye(3) if R is None else R, key=None)]


def _dihedral(subunits):
    """Add the partners generated by a 2-fold axis along x (dihedral symmetry)."""
    Rx = _rx(np.pi)
    return subunits + [dict(radii=s["radii"], centre=tuple(Rx @ np.array(s["centre"])), R=Rx @ s["R"], key=s["key"])
                       for s in subunits]


def _blueprint(pid):
    """Subunit layout (nm) of the code-generated stand-in for each zoo entry."""
    if pid in ("4v4r", "4v7r", "5mrc"):
        s = {"4v4r": 1.0, "4v7r": 1.13, "5mrc": 1.14}[pid]
        L = [_lobe((9.0 * s, 8.0 * s, 6.5 * s), (0, 0, 4.5 * s)),                       # large subunit body
             _lobe((3.0 * s, 3.0 * s, 3.0 * s), (0, 1.0 * s, 11.0 * s)),                # central protuberance
             _lobe((2.0 * s, 2.0 * s, 3.5 * s), (-9.0 * s, 0, 7.0 * s), _ry(-0.6)),     # L1-like stalk
             _lobe((1.4 * s, 1.4 * s, 4.0 * s), (9.5 * s, 0, 6.5 * s), _ry(0.7)),       # L7/L12-like stalk
             _lobe((7.5 * s, 5.0 * s, 3.8 * s), (1.0 * s, 0, -4.5 * s)),                # small subunit body
             _lobe((3.5 * s, 3.2 * s, 3.0 * s), (-5.5 * s, -1.0 * s, -6.5 * s))]        # small subunit head
        if pid == "4v7r":                                                                # eukaryotic expansion segments
            L += [_lobe((3.2, 2.6, 2.8), (8.0, 7.0, 7.0)), _lobe((3.0, 2.6, 3.2), (-6.0, -8.0, -6.5)),
                  _lobe((2.4, 2.4, 3.6), (-10.5, 4.0, -1.0), _ry(0.5))]
        if pid == "5mrc":                                                                # mitochondria-specific proteins
            L += [_lobe((4.2, 3.6, 2.8), (0, -7.5, 12.5)), _lobe((3.4, 3.0, 3.4), (9.0, -6.0, -8.0)),
                  _lobe((3.0, 3.0, 3.0), (-9.5, 7.0, 1.0)), _lobe((2.5, 4.5, 2.0), (2.0, 9.0, 9.0))]
        prng = np.random.default_rng(_seed(pid) + 1)                                     # rRNA-like surface bumps
        for _ in range(8):
            d = prng.normal(size=3); d /= np.linalg.norm(d)
            L += [_lobe(tuple(prng.uniform(1.2, 1.9, 3)), tuple(s * d * np.array([9.5, 8.0, 10.5])), _rz(prng.uniform(0, 6)))]
        return [x for l in L for x in l]
    if pid in ("3j9i", "6utj", "4cr2"):
        core = []
        for z, radii, ph in [(1.9, (1.9, 1.7, 2.0), 0.0), (5.6, (2.1, 1.7, 1.85), np.pi / 7)]:
            core += _ring(7, 3.6, z, radii, ph)
        core = _dihedral(core)                                                           # α7β7β7α7, D7
        if pid == "6utj":
            core += _ring(7, 3.3, 10.0, (1.9, 1.6, 2.4), np.pi / 14)                     # one heptameric cap: C7
        if pid == "4cr2":
            core += _ring(6, 4.3, 9.6, (2.2, 1.8, 2.0), 0.3)                             # base ring of six
            core += (_lobe((4.0, 3.0, 3.0), (3.5, 0, 14.5)) + _lobe((3.0, 2.5, 3.5), (-1.5, 3.0, 15.5))
                     + _lobe((2.5, 2.5, 2.5), (-3.5, -2.0, 13.5)) + _lobe((3.0, 3.0, 2.5), (-4.5, 0.5, 10.5)))
        return core
    if pid == "4v94":
        one = []
        for z in (3.6, -3.6):
            one += _ring(8, 5.2, z, (2.6, 2.0, 3.4), np.pi / 8 if z < 0 else 0.0)
        two = []
        for dx, rot in [(-8.5, 0.0), (8.5, np.pi)]:
            for s in one:
                c = _rz(rot) @ np.array(s["centre"])
                two.append(dict(radii=s["radii"], centre=(c[0] + dx, c[1], c[2]), R=_rz(rot) @ s["R"], key=s["key"]))
        return two
    if pid == "1bxn":
        # like the deposited file (the crystal's asymmetric unit): one L4S4 layer = half of the L8S8 enzyme, C4.
        # `_dihedral` of this list would give the complete enzyme (D4).
        return _ring(4, 3.6, 2.2, (2.6, 2.4, 2.4)) + _ring(4, 2.6, 5.0, (1.9, 1.7, 1.35), np.pi / 4)
    if pid == "1qvr":
        # like the deposited file: three neighbouring subunits (half of the C6 hexameric ring), C1
        tiers = [_ring(6, 4.2, 2.2, (2.2, 2.0, 2.0)), _ring(6, 3.9, -2.2, (2.3, 2.0, 2.0), np.pi / 6),
                 _ring(6, 7.2, 2.4, (0.9, 0.9, 4.6), 0.4, tilt=_rx(1.2)),               # coiled-coil 'propellers'
                 _ring(6, 3.8, 6.6, (1.7, 1.6, 1.8), 0.5)]                               # N-domains on top
        return [s for t in tiers for s in t[:3]]
    if pid == "3cf3":
        return (_ring(6, 4.0, 2.0, (2.1, 2.0, 2.0)) + _ring(6, 4.0, -2.2, (2.2, 2.0, 2.1), np.pi / 6)
                + _ring(6, 6.6, 3.0, (2.0, 1.6, 1.8), 0.25))                             # N-domains flush: C6
    if pid == "2cg9":
        half = (_lobe((1.9, 1.7, 1.7), (2.2, 0.8, 6.0)) + _lobe((2.2, 1.8, 2.0), (2.6, -0.4, 2.0))
                + _lobe((1.7, 1.5, 1.8), (1.4, 0.3, -2.2)) + _lobe((1.4, 1.2, 1.2), (4.2, 1.6, 7.6)))
        Rz = _rz(np.pi)
        return half + [dict(radii=h["radii"], centre=tuple(Rz @ np.array(h["centre"])), R=Rz @ h["R"], key=None) for h in half]
    if pid == "1u6g":
        out = []
        for k in range(6):                                                                # curved cullin stalk
            t = k / 5
            out += _lobe((1.4, 1.4, 1.6), (8.0 * np.sin(0.9 * t) - 3.0, 0, 9.0 * t - 4.5), _ry(0.9 * t))
        out += _lobe((2.2, 2.0, 2.0), (4.0, 0, 5.8)) + _lobe((1.2, 1.1, 1.1), (6.0, 1.8, 6.5))
        for k in range(10):                                                               # Cand1-like superhelix
            a = 1.6 * np.pi * k / 9
            out += _lobe((1.5, 1.4, 1.4), (-3.0 + 7.0 * np.sin(0.9 * k / 9) + 2.6 * np.cos(a), 2.6 * np.sin(a), 9.0 * k / 9 - 4.5))
        return out
    if pid == "3d2f":
        comp = (_lobe((1.8, 1.6, 1.8), (0, 0, 0)) + _lobe((1.8, 1.6, 1.8), (0, 3.2, 0)) + _lobe((1.6, 1.4, 2.4), (3.0, 1.6, 1.8))
                + _lobe((1.6, 1.5, 1.6), (-3.2, 1.0, -1.0)) + _lobe((1.6, 1.5, 1.6), (-3.2, 3.6, -0.6)))
        R2, t2 = _rx(2.3) @ _rz(0.7), np.array([1.5, -4.5, 5.5])
        return comp + [dict(radii=c["radii"], centre=tuple(R2 @ np.array(c["centre"]) + t2), R=R2 @ c["R"], key=None) for c in comp]
    if pid == "1s3x":
        return (_lobe((1.4, 1.4, 1.6), (-1.6, 0, -1.2)) + _lobe((1.3, 1.3, 1.5), (-1.4, 0.2, 1.6))
                + _lobe((1.4, 1.4, 1.6), (1.6, 0, -1.2)) + _lobe((1.2, 1.3, 1.4), (1.5, -0.3, 1.8)))
    raise KeyError(pid)


def _seed(pid):
    return int.from_bytes(pid.encode(), "little") % (2 ** 31)


_SCALE = {"4v4r": 0.923, "4v7r": 0.906, "5mrc": 0.883, "3j9i": 0.930, "6utj": 0.930, "4cr2": 0.930, "4v94": 0.886,
          "1bxn": 0.930, "1qvr": 0.885, "3cf3": 0.935, "2cg9": 0.995, "1u6g": 0.934, "3d2f": 0.997, "1s3x": 0.884}
_ATOMS_CACHE = {}
def zoo_atoms(pid):
    """Atomic model (dict of arrays) of the code-generated stand-in for zoo entry `pid` (Å, centred at the origin)."""
    pid = pid.lower()
    if pid in _ATOMS_CACHE:
        return _ATOMS_CACHE[pid]
    rng = np.random.default_rng(_seed(pid))
    built, parts = {}, []
    f = _SCALE.get(pid, 1.0)
    for k, s in enumerate(_blueprint(pid)):
        radii_A = tuple(10.0 * f * r for r in s["radii"])
        key = s["key"]
        if key is not None and key in built:
            sub = built[key]
        else:
            sub = helix_bundle(radii_A, rng)
            if key is not None:
                built[key] = sub
        part = _transform(sub, s["R"], 10.0 * f * np.array(s["centre"]))
        part["chain"] = np.full(len(part["xyz"]), _CHAIN_IDS[k % len(_CHAIN_IDS)])
        parts.append(part)
    atoms = merge_atoms(parts)
    if pid in ("3j9i", "6utj", "4cr2"):                             # lay the proteasome barrels along x (side view down z)
        atoms["xyz"] = (atoms["xyz"] @ _ry(np.pi / 2).T).astype(np.float32)
    atoms["xyz"] -= atoms["xyz"].mean(0)
    _ATOMS_CACHE[pid] = atoms
    return atoms


_Z = {"C": 6.0, "N": 7.0, "O": 8.0, "S": 16.0, "P": 15.0, "H": 1.0}
def atoms_to_map(xyz, weights=None, voxel=10.0, box=None, sigma_A=None):
    """Density map [z, y, x] from atom positions (Å): trilinear splatting on a grid twice as fine, Gaussian blur
    (σ = `sigma_A`, default = voxel/2), then 2×2×2 binning to `voxel` Å. Centred on the box if `box` is given."""
    xyz = np.asarray(xyz, float)
    w = np.ones(len(xyz)) if weights is None else np.asarray(weights, float)
    fine = voxel / 2.0
    if box is None:
        ext = np.ptp(xyz, 0).max() + 6 * voxel
        box = int(np.ceil(ext / voxel)) + 1
    n = 2 * box
    centre = (n - 1) / 2.0
    p = xyz[:, ::-1] / fine + centre                                  # (z, y, x) grid coordinates
    grid = np.zeros((n, n, n), np.float32)
    i0 = np.floor(p).astype(int)
    f = p - i0
    for dz in (0, 1):
        for dy in (0, 1):
            for dx in (0, 1):
                idx = i0 + np.array([dz, dy, dx])
                wt = w * np.where(dz, f[:, 0], 1 - f[:, 0]) * np.where(dy, f[:, 1], 1 - f[:, 1]) * np.where(dx, f[:, 2], 1 - f[:, 2])
                ok = np.all((idx >= 0) & (idx < n), 1)
                np.add.at(grid, tuple(idx[ok].T), wt[ok])
    grid = ndimage.gaussian_filter(grid, (sigma_A if sigma_A else voxel / 2.0) / fine)
    return grid.reshape(box, 2, box, 2, box, 2).mean((1, 3, 5)).astype(np.float32)


def zoo_map(pid, voxel="10A"):
    """(density[z, y, x], voxel size Å, source) of the code-generated stand-in for a zoo entry (no download)."""
    atoms = zoo_atoms(pid)
    v = float(str(voxel).rstrip("A"))
    w = np.array([_Z.get(e, 6.0) for e in atoms["element"]])
    vol = atoms_to_map(atoms["xyz"], w, voxel=v)
    return vol, v, f"code-generated stand-in modelled on PDB {pid.upper()}"


def betagal_like_map(voxel=3.54, box=100, b_factor=40.0):
    """A β-galactosidase-sized D2 homotetramer (code-generated, ~4 × 1,000 residues) at `voxel` Å, `box`³ voxels.

    Returns (map[z, y, x], voxel, source, atoms). Atomic B-factor `b_factor` (Å²) blurs each atom by
    σ = √(B / 8π²); helices (≈ 10 Å apart) are visible at this sampling, side chains are not.
    """
    rng = np.random.default_rng(1234)
    domains = [((21.0, 18.0, 22.0), (0.0, 0.0, 0.0)), ((18.0, 16.0, 20.0), (34.0, 8.0, 4.0)),
               ((17.0, 15.0, 19.0), (-31.0, 10.0, -3.0)), ((16.0, 15.0, 18.0), (6.0, 31.0, 8.0)),
               ((15.0, 14.0, 17.0), (-8.0, -29.0, 5.0))]
    parts = [_transform(helix_bundle(r, rng), _rz(rng.uniform(0, 6.3)), c) for r, c in domains]
    mono = merge_atoms(parts)
    mono["xyz"] = mono["xyz"] - mono["xyz"].mean(0) + np.array([30.0, 21.0, 13.0])
    subunits = []
    for k, R in enumerate([np.eye(3), _rz(np.pi), _rx(np.pi), _ry(np.pi)]):          # D2: identity + three 2-folds
        s = _transform(mono, R)
        s["chain"] = np.full(len(s["xyz"]), "ABCD"[k])
        subunits.append(s)
    atoms = merge_atoms(subunits)
    atoms["xyz"] -= atoms["xyz"].mean(0)
    w = np.array([_Z.get(e, 6.0) for e in atoms["element"]])
    sigma = np.sqrt(b_factor / (8 * np.pi ** 2) + (voxel / 2.0) ** 2)                  # B-factor + sampling blur
    vol = atoms_to_map(atoms["xyz"], w, voxel=voxel, box=box, sigma_A=sigma)
    return vol, voxel, "code-generated β-galactosidase-sized D2 tetramer", atoms


def prepare_map(vol: np.ndarray, size: int = 32, sigma: float = 1.0) -> np.ndarray:
    """Low-pass, centre (by centre of mass), crop/pad to ``size``³ and scale to [0, 1].

    Zoo maps are sharp sums of atoms; a Gaussian blur of ~1 voxel (1 nm)
    mimics the resolution of a cellular tomogram.
    """
    v = ndimage.gaussian_filter(np.clip(vol, 0, None).astype(np.float32), sigma)
    s = np.array(v.shape)
    src0 = np.maximum((s - size) // 2, 0)                  # where the crop starts in the input ...
    dst0 = np.maximum((size - s) // 2, 0)                  # ... and where the pad starts in the output
    n = np.minimum(s, size)
    com = np.array(ndimage.center_of_mass(v))
    shift = (size - 1) / 2 - dst0 + src0 - com             # puts the centre of mass exactly at the output's centre
    v = ndimage.shift(v, shift, order=1)
    out = np.zeros((size, size, size), np.float32)
    out[dst0[0]:dst0[0] + n[0], dst0[1]:dst0[1] + n[1], dst0[2]:dst0[2] + n[2]] = \
        v[src0[0]:src0[0] + n[0], src0[1]:src0[1] + n[1], src0[2]:src0[2] + n[2]]
    out -= out.min()
    return out / (out.max() + 1e-12)


def random_rotation(rng: np.random.Generator) -> np.ndarray:
    """Uniformly random proper rotation (det = +1) from a random unit quaternion."""
    q = rng.normal(size=4)
    q /= np.linalg.norm(q)
    w, x, y, z = q
    return np.array([
        [1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
        [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
        [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])


_P = np.array([[0, 0, 1], [0, 1, 0], [1, 0, 0]], float)  # (x,y,z) <-> (z,y,x)
def rotate_volume(vol: np.ndarray, R: np.ndarray, shift_xyz=(0, 0, 0), order: int = 1,
                  cval: float = 0.0) -> np.ndarray:
    """Rotate a volume about its centre by R (then translate by shift, in voxels, (x,y,z)).

    out(p) = vol(Rᵀ (p − c − t) + c): the *object* is rotated by R and moved by t.
    """
    R_idx = _P @ R @ _P                       # same rotation written in (z, y, x) index order
    c = (np.array(vol.shape) - 1) / 2.0
    t_idx = _P @ np.asarray(shift_xyz, float)
    M = R_idx.T                               # inverse rotation maps output -> input
    offset = c - M @ (c + t_idx)
    return ndimage.affine_transform(vol, M, offset=offset, order=order, mode="constant", cval=cval)


def normalize(v: np.ndarray) -> np.ndarray:
    """Zero mean, unit standard deviation."""
    return ((v - v.mean()) / (v.std() + 1e-12)).astype(np.float32)


def match_points(pred, true, radius: float):
    """Greedy one-to-one matching of predicted to true coordinates within ``radius``.

    Returns dict(precision, recall, f1, tp, fp, fn).
    """
    true = np.asarray(true, float)
    pred = np.asarray(pred, float).reshape(-1, true.shape[-1] if true.ndim == 2 else 3)
    if len(pred) == 0 or len(true) == 0:
        tp = 0
    else:
        d = np.linalg.norm(pred[:, None, :] - true[None, :, :], axis=-1)
        used_p, used_t, tp = set(), set(), 0
        for i, j in zip(*np.unravel_index(np.argsort(d, axis=None), d.shape)):
            if d[i, j] > radius:
                break
            if i not in used_p and j not in used_t:
                used_p.add(i); used_t.add(j); tp += 1
    fp, fn = len(pred) - tp, len(true) - tp
    p = tp / max(len(pred), 1)
    r = tp / max(len(true), 1)
    return dict(precision=p, recall=r, f1=2 * p * r / max(p + r, 1e-12), tp=tp, fp=fp, fn=fn)
print("🧰 toolkit loaded")

---
## 1 · The sparse-label problem

📍 **Where we are.** Module 11 picked particles without any training (template matching, DoG); Module 19 learned
classifiers from few labels when every labelled example had a *class*. Picking adds a twist: the labels are a few
**positive locations**, and everything else is **unlabelled** — a mixture of background and unclicked particles.

Our toy is a 2D "slab" through a crowded cell: projections of ribosomes (our targets) mixed with 20S proteasomes,
capped proteasomes and RuBisCO (not targets), plus noise. An "expert" has clicked **10** ribosomes in a training image;
a second image of the same kind serves as the test.

In [ ]:
# ── 1.1 Two crowded 2D slabs: training and test ──
CAST = ["4v4r", "3j9i", "6utj", "1bxn"]
tmpl = {}
for pid in CAST:
    vol, vs, src = zoo_map(pid)
    tmpl[pid] = prepare_map(vol, size=32, sigma=1.0)
    print(f"{pid.upper()} {ZOO[pid]['name']:24s} ← {src}")
H = 256                                                     # image size (pixels of 1 nm)
R_PART = 7                                                  # radius (pixels) of a labelled ribosome disk


def crowded_slab(rng, n_ribo=70, n_other=60, snr=0.3):
    """2D slab: ribosome projections (targets) + other complexes, non-overlapping, plus white noise.
    Returns (normalised image, ribosome centres (y, x))."""
    img, centres, occupied = np.zeros((H, H), np.float32), [], []
    order = ["4v4r"] * n_ribo + ["3j9i", "6utj", "1bxn"] * (n_other // 3)
    rng.shuffle(order)
    for pid in order:
        r = 13 if pid == "4v4r" else 9
        for _ in range(200):
            c = rng.uniform(16, H - 16, 2)
            if all(np.linalg.norm(c - o) > r + ro for o, ro in occupied):
                break
        else:
            continue                                         # no room left for this one
        y0, x0 = (c - 16).astype(int)
        img[y0:y0 + 32, x0:x0 + 32] += rotate_volume(tmpl[pid], random_rotation(rng)).sum(0)
        occupied.append((c, r))
        if pid == "4v4r":
            centres.append(c)
    noisy = ndimage.gaussian_filter(img, 0.5) + rng.normal(0, np.sqrt(img.var() / snr), img.shape)
    return normalize(noisy), np.array(centres)


train_img, train_centres = crowded_slab(rng)
test_img, test_centres = crowded_slab(rng)
N_LAB = 10
labelled = train_centres[rng.choice(len(train_centres), N_LAB, replace=False)]
yy, xx = np.mgrid[:H, :H]
P_mask = np.zeros((H, H), bool)
for c in labelled:
    P_mask |= (yy - c[0]) ** 2 + (xx - c[1]) ** 2 <= R_PART ** 2
print(f"training slab: {len(train_centres)} ribosomes, {N_LAB} labelled ({100 * P_mask.mean():.2f} % of pixels positive)")
print(f"test slab: {len(test_centres)} ribosomes")
fig, ax = plt.subplots(1, 2, figsize=(9, 4.4))
ax[0].imshow(train_img); ax[0].set_title("training slab: 10 clicked ribosomes (red)")
ax[0].scatter(labelled[:, 1], labelled[:, 0], s=120, facecolors="none", edgecolors="tab:red")
ax[1].imshow(train_img); ax[1].set_title("…and all ribosomes really there (cyan)")
ax[1].scatter(train_centres[:, 1], train_centres[:, 0], s=120, facecolors="none", edgecolors="tab:cyan")
for a in ax:
    a.axis("off")
plt.tight_layout(); plt.show()

---
## 2 · Positive–unlabelled learning

### 2.1 Picking as pixel classification

TomoPicker frames localisation as **voxel classification**: each voxel is "inside a particle" (positive) or not. Labelled
positives $\mathcal P$ are spheres around the clicked centres; everything else is **unlabelled** $\mathcal U$. To keep this
section fast and transparent we use a simple, *convex* classifier — logistic regression on nine per-pixel features
(Gaussian-smoothed intensities at six scales and three difference-of-Gaussians responses, Module 11). TomoPicker uses a
3D residual U-Net; the learning principles below are the same.

In [ ]:
# ── 2.1 Per-pixel features and a logistic classifier trained with different objectives ──
def pixel_features(img):
    """Nine standardised features per pixel: Gaussian blurs at 6 scales and 3 difference-of-Gaussians."""
    g = [ndimage.gaussian_filter(img, s) for s in (1, 2, 3, 4, 6, 8)]
    f = np.stack(g + [g[1] - g[3], g[2] - g[4], g[3] - g[5]], -1).reshape(-1, 9)
    return torch.tensor((f - f.mean(0)) / f.std(0), dtype=torch.float32)


F_train, F_test = pixel_features(train_img), pixel_features(test_img)
is_pos = torch.tensor(P_mask.reshape(-1))
bce = nn.BCEWithLogitsLoss()
print(f"{F_train.shape[0]:,} training pixels × {F_train.shape[1]} features")

### 2.2 Naive positive–negative (PN) training — and what it really learns

The naive approach labels $\mathcal P$ as 1 and **all of** $\mathcal U$ as 0. Under a classic assumption (the clicked
particles are a random sample of all particles), Elkan & Noto (2008) showed what such a classifier learns:

$$P(\text{labelled}\mid x)=c\cdot P(\text{particle}\mid x),\qquad c=\frac{\#\text{labelled particles}}{\#\text{all particles}} .$$

*In words:* its probabilities are the true ones **shrunk by the labelling fraction** $c$ (here ≈ 10/31). The **ranking**
of pixels can still be right, but the **calibration** is wrong: a threshold of 0.5 finds almost nothing. With a flexible
deep network there is a second problem — it can *memorise* the ten clicked particles and push all unclicked ribosomes
toward "negative", damaging the ranking too.

In [ ]:
# ── 2.2 Fitting helper and the naive PN objective ──
def fit(objective, steps=400, lr=0.05):
    """Fit a logistic model w·x + b by full-batch Adam under the given objective. Returns a scoring function."""
    torch.manual_seed(SEED)
    w = torch.zeros(F_train.shape[1], requires_grad=True)
    b = torch.zeros(1, requires_grad=True)
    opt = torch.optim.Adam([w, b], lr=lr)
    for _ in range(steps):
        logits = F_train @ w + b
        loss = objective(logits[is_pos], logits[~is_pos])
        opt.zero_grad(); loss.backward(); opt.step()
    return lambda feats: torch.sigmoid(feats @ w.detach() + b.detach()).numpy().reshape(H, H)


def pn_objective(lp, lu):
    """Positive–negative: clicked pixels = 1, every other pixel = 0 (ordinary binary cross-entropy)."""
    return bce(torch.cat([lp, lu]), torch.cat([torch.ones_like(lp), torch.zeros_like(lu)]))


print("PN objective ready")

### 2.3 The class prior π from an expected count

Both PU methods need the fraction of unlabelled pixels that are really positive. TomoPicker computes it from two numbers a
biologist can estimate — the expected number of particles $E_p$ and the volume (here area) of one particle $R_n$:

$$\pi' = \frac{E_p\times R_n}{|\mathcal P\cup\mathcal U|}.$$

Below, $\pi_u$ is the same quantity restricted to the unlabelled pixels (it excludes the clicked particles). We give the
method the true count of the training slab; in practice it is a rough estimate (✏️ try a wrong one later).

In [ ]:
# ── 2.3 Class priors ──
E_p = len(train_centres)                                      # expected number of ribosomes (an estimate in practice)
R_n = np.pi * R_PART ** 2                                     # pixels per particle disk
pi_all = E_p * R_n / H ** 2
pi_u = (E_p - N_LAB) * R_n / float((~is_pos).sum())
print(f"π' = {pi_all:.3f} of all pixels are expected inside particles;  π_u = {pi_u:.3f} of the unlabelled pixels")

### 2.4 Non-negative PU risk (nnPU) and generalized expectation (GE-KL)

**nnPU** (Kiryo *et al.* 2017) rewrites the ordinary classification risk using only positive and unlabelled data. With
$\hat R^+_P$ the loss of calling positives positive, $\hat R^-_P$ the loss of calling positives negative, and $\hat R^-_U$ the
loss of calling unlabelled pixels negative,

$$\hat R_{PU}=\pi\,\hat R^+_P+\max\!\big(0,\ \hat R^-_U-\pi\,\hat R^-_P\big).$$

*In words:* "calling unlabelled pixels negative" is penalised **except** for the part explained by the positives hidden
among them ($\pi\hat R^-_P$). The $\max(0,\cdot)$ keeps this estimated negative risk from going below zero — when it would,
TomoPicker's code takes a gradient step that pushes it back up.

**GE-KL** (generalized expectation) instead says: fit the positives, and make the **average prediction on unlabelled
pixels match π**:

$$\mathcal L=\mathbb E_{x\in\mathcal P}\big[\mathrm{BCE}(f(x),1)\big]+\lambda\,\mathrm{KL}\big(\pi\ \big\|\ \bar p_U\big),\qquad \bar p_U=\mathbb E_{x\in\mathcal U}\,\sigma(f(x)),\quad\lambda=10,$$

with the Bernoulli KL divergence $\mathrm{KL}(\pi\|\bar p)=\pi\log\frac{\pi}{\bar p}+(1-\pi)\log\frac{1-\pi}{1-\bar p}$.

In [ ]:
# ── 2.4 nnPU and GE-KL objectives ──
def nnpu_objective(lp, lu, pi=pi_u):
    """Non-negative PU risk with the logistic loss; flips sign when the negative-risk estimate drops below 0."""
    r_p_pos = bce(lp, torch.ones_like(lp))
    r_p_neg = bce(lp, torch.zeros_like(lp))
    r_u_neg = bce(lu, torch.zeros_like(lu))
    neg = r_u_neg - pi * r_p_neg
    return -neg if neg.item() < 0 else pi * r_p_pos + neg


def gekl_objective(lp, lu, pi=pi_u, lam=10.0):
    """Generalized expectation: BCE on positives + λ · KL(π ‖ mean prediction on unlabelled)."""
    p_u = torch.sigmoid(lu).mean().clamp(1e-6, 1 - 1e-6)
    kl = pi * torch.log(pi / p_u) + (1 - pi) * torch.log((1 - pi) / (1 - p_u))
    return bce(lp, torch.ones_like(lp)) + lam * kl


def evaluate(score_map, centres, k_expected):
    """Picks = local maxima (15-pixel window). Report: mean probability, F1 at threshold 0.5, F1 of the top-k peaks."""
    peaks = np.argwhere(score_map == ndimage.maximum_filter(score_map, size=15))
    peaks = peaks[np.argsort(-score_map[peaks[:, 0], peaks[:, 1]])]
    thr = match_points(peaks[score_map[peaks[:, 0], peaks[:, 1]] > 0.5], centres, R_PART)
    top = match_points(peaks[:k_expected], centres, R_PART)
    return dict(mean_p=float(score_map.mean()), n_thr=thr["tp"] + thr["fp"], f1_thr=thr["f1"], f1_topk=top["f1"])


t0 = time.time()
scores, results = {}, {}
for name, obj in [("PN (unlabelled = negative)", pn_objective), ("nnPU", nnpu_objective), ("GE-KL", gekl_objective)]:
    model = fit(obj)
    scores[name] = model(F_test)
    results[name] = evaluate(scores[name], test_centres, k_expected=len(test_centres))
print(f"three fits in {time.time() - t0:.1f} s; true fraction of particle pixels in the test slab ≈ "
      f"{len(test_centres) * R_n / H**2:.3f}")
print(f"{'objective':28s} {'mean p':>7s} {'picks>0.5':>10s} {'F1 @0.5':>8s} {'F1 top-k':>9s}")
for name, r in results.items():
    print(f"{name:28s} {r['mean_p']:7.3f} {r['n_thr']:10d} {r['f1_thr']:8.2f} {r['f1_topk']:9.2f}")

In [ ]:
# ── 2.5 Probability maps ──
fig, axes = plt.subplots(1, 4, figsize=(12, 3.3))
axes[0].imshow(test_img); axes[0].set_title("test slab"); axes[0].axis("off")
for ax, (name, s) in zip(axes[1:], scores.items()):
    im = ax.imshow(s, cmap="magma", vmin=0, vmax=1); ax.axis("off")
    ax.set_title(f"{name}\nF1 at p > 0.5: {results[name]['f1_thr']:.2f}", fontsize=8)
plt.colorbar(im, ax=axes[1:], fraction=0.02, label="P(particle)")
plt.show()

🔍 **What just happened?** All three classifiers *rank* pixels well — taking the expected number of strongest peaks
finds the ribosomes in every case (F1 top-k). But only the PU objectives are even roughly **calibrated**: their average
probability is of the same order as the true particle fraction, so a plain 0.5 threshold gives sensible picks — nnPU
almost perfectly, GE-KL with some over-picking here. The PN model's probabilities are shrunk by the labelling fraction
(Section 2.2), so at 0.5 it finds only a handful of ribosomes. With deep networks the damage goes
further: in the TomoPicker paper, a strong supervised picker (DeepETPicker) trained on 10 labels reached F1 0.17 on
*S. pombe* tomograms, against 0.46 for TomoPicker (GE-KL) — while DeepETPicker needed ~500 labels to reach 0.57.

✏️ **Your turn.** Give the PU methods a wrong expected count (`E_p = 15` or `E_p = 60` in 2.3) and re-run 2.3–2.5.
How sensitive are they to a bad estimate of π?

---
## 3 · CryoMAE-lite: few-shot picking with a masked autoencoder

Single-particle cryo-EM has the same bottleneck in 2D micrographs. **CryoMAE** (Xu, Zhan & Xu, WACV 2025) needs only
about **15 exemplar particles**:

1. train a **masked autoencoder** (MAE) — a network that sees a crop with 75 % of its patches hidden and must reconstruct
   them — on the exemplars and random crops, so it learns what micrograph patches look like;
2. add a **self–cross similarity loss** that pulls exemplar embeddings together and away from random crops,

$$S_{self}=\tfrac{1}{m^2}\sum_{i,j}\cos\big(E(\hat x^l_i),E(\hat x^l_j)\big),\quad S_{cross}=\tfrac{1}{mn}\sum_{i,j}\cos\big(E(\hat x^l_i),E(\hat x^u_j)\big),\quad L_{SCS}=\max\big(\tau,\ 1+\alpha S_{cross}-(1-\alpha)S_{self}\big);$$

3. slide the encoder over new micrographs and score each window by its **maximum** cosine similarity to the exemplars;
4. keep the candidates above an **elbow** in the scores.

Our miniature uses the β-galactosidase-sized map of Modules 01, 06 and 10 to make particle images, adds small "junk" blobs, and replaces the paper's
ViT with a tiny MLP over 8×8 patches. The paper's values of α, τ and β, and its positive-unlabelled adjustment of
$L_{SCS}$, were not available to us; we use α = 0.5, τ = 0, β = 1.

In [ ]:
# ── 3.1 Micrographs of β-galactosidase with junk ──
bgal, bgal_px, src, _ = betagal_like_map()                       # code-generated, 100³ at 3.54 Å/voxel
bgal = np.clip(bgal.reshape(50, 2, 50, 2, 50, 2).mean((1, 3, 5)), 0, None)   # bin 2× → 50³ at ~7 Å/pixel
bgal /= bgal.max()
print("particle map:", src)
HM, C = 320, 32                                                   # micrograph size, crop size


def micrograph(rng, snr, n=45, n_junk=25):
    """Projections of β-galactosidase in random orientations + small junk blobs + noise."""
    img, centres = np.zeros((HM, HM), np.float32), []
    for _ in range(n):
        for _ in range(300):
            c = rng.uniform(24, HM - 24, 2)
            if all(np.linalg.norm(c - o) > 30 for o in centres):
                y0, x0 = (c - 25).astype(int)
                img[y0:y0 + 50, x0:x0 + 50] += rotate_volume(bgal, random_rotation(rng)).sum(0)
                centres.append(c)
                break
    yy_, xx_ = np.mgrid[:HM, :HM]
    for _ in range(n_junk):
        c = rng.uniform(10, HM - 10, 2)
        img += 1.5 * np.exp(-((yy_ - c[0]) ** 2 + (xx_ - c[1]) ** 2) / (2 * 3**2))
    noisy = img + rng.normal(0, np.sqrt(img.var() / snr), img.shape)
    return normalize(noisy), np.array(centres)


def crop_at(img, c):
    y, x = np.round(c).astype(int)
    return img[y - C // 2:y + C // 2, x - C // 2:x + C // 2]


mg_rng = np.random.default_rng(1)
micro = {snr: (micrograph(mg_rng, snr), micrograph(mg_rng, snr)) for snr in (0.1, 0.05)}
(ref_img, ref_c), (qry_img, qry_c) = micro[0.1]
fig, axes = plt.subplots(1, 2, figsize=(9, 4.3))
axes[0].imshow(ref_img); axes[0].set_title("reference micrograph (SNR 0.1): 15 exemplars clicked")
axes[0].scatter(ref_c[:15, 1], ref_c[:15, 0], s=160, facecolors="none", edgecolors="tab:red")
axes[1].imshow(micro[0.05][1][0]); axes[1].set_title("a query micrograph at SNR 0.05")
for a in axes:
    a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# ── 3.2 A miniature masked autoencoder over 8×8 patches ──
PATCH, NP_ = 8, (C // 8) ** 2


def patchify(x):
    """(B, 32, 32) → (B, 16 patches, 64 pixels)."""
    B = x.shape[0]
    return x.reshape(B, C // PATCH, PATCH, C // PATCH, PATCH).permute(0, 1, 3, 2, 4).reshape(B, NP_, PATCH * PATCH)


class TinyMAE(nn.Module):
    """Patch embedding + position embedding → MLP encoder; decoder reconstructs hidden patches from visible context."""

    def __init__(self, d=64):
        super().__init__()
        self.embed_patch = nn.Linear(PATCH * PATCH, d)
        self.pos = nn.Parameter(torch.randn(NP_, d) * 0.02)
        self.enc = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Linear(d, d))
        self.dec = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Linear(d, PATCH * PATCH))
        self.mask_token = nn.Parameter(torch.zeros(d))

    def reconstruction_loss(self, x, mask_ratio=0.75):
        p = patchify(x)
        keep = (torch.rand(p.shape[:2]) > mask_ratio).float()          # 1 = visible patch
        h = self.enc((self.embed_patch(p) + self.pos) * keep[..., None])
        context = (h * keep[..., None]).sum(1, keepdim=True) / keep.sum(1)[:, None, None].clamp(min=1)
        h_dec = torch.where(keep[..., None] > 0, h, self.mask_token + context + self.pos)
        err = ((self.dec(h_dec) - p) ** 2).mean(-1)
        return (err * (1 - keep)).sum() / (1 - keep).sum().clamp(min=1)  # MSE on the hidden patches only

    def embedding(self, x):
        """Unit-length crop embedding (no masking): mean of the encoded patches."""
        return F.normalize(self.enc(self.embed_patch(patchify(x)) + self.pos).mean(1), dim=-1)


def train_mae(exemplars, randoms, use_scs, steps=400, alpha=0.5, tau=0.0, beta=1.0):
    """MAE pre-training on exemplars + random crops, optionally with the self–cross similarity loss."""
    torch.manual_seed(SEED)
    m = TinyMAE(); opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    E, U = torch.tensor(exemplars), torch.tensor(randoms)
    for _ in range(steps):
        idx = torch.randint(len(U), (48,))
        xb = torch.rot90(torch.cat([E, U[idx]]), int(torch.randint(4, (1,))), (1, 2))   # rotation augmentation
        loss = m.reconstruction_loss(xb)
        if use_scs:
            ze, zu = m.embedding(E), m.embedding(U[idx])
            s_self, s_cross = (ze @ ze.T).mean(), (ze @ zu.T).mean()
            loss = loss + beta * torch.clamp(1 + alpha * s_cross - (1 - alpha) * s_self, min=tau)
        opt.zero_grad(); loss.backward(); opt.step()
    return m


print("TinyMAE ready")

### 3.3 Sliding-window scoring and the elbow threshold

For each 32×32 window of the query micrograph (stride 4 pixels) we compute its embedding and score it by its **maximum**
cosine similarity to the 15 exemplar embeddings (the paper found max better than mean). After non-maximum suppression the
candidates are sorted by score, and we cut at the **elbow** — the point where the scores drop most sharply. *(The paper
places the elbow on the rate of change of the mean distance to the k = 5 nearest neighbours; ours is a simplified
version on the sorted scores.)*

We compare three training set-ups at two noise levels:

* **MAE only** — reconstruction loss, no similarity loss;
* **MAE + SCS, random crops** — the similarity loss with random crops of the micrograph playing the "other" role;
* **MAE + SCS, clean crops (oracle)** — the same, but the "other" crops are chosen, *using the hidden truth*, to contain
  no particle. This is a **control experiment**: no real user could do it, but it tells us what the loss could achieve if
  its negatives were really negative.

In [ ]:
# ── 3.3 Picking with exemplars ──
def pick_with_exemplars(m, img, exemplars, stride=4):
    """Return candidate coordinates sorted by score, their scores, and the elbow index."""
    ys = np.arange(C // 2, HM - C // 2, stride); xs = np.arange(C // 2, HM - C // 2, stride)
    crops = np.stack([img[y - C // 2:y + C // 2, x - C // 2:x + C // 2] for y in ys for x in xs])
    with torch.no_grad():
        z = torch.cat([m.embedding(torch.tensor(crops[i:i + 512])) for i in range(0, len(crops), 512)])
        s = (z @ m.embedding(torch.tensor(exemplars)).T).max(1).values.numpy().reshape(len(ys), len(xs))
    pk = np.argwhere(s == ndimage.maximum_filter(s, size=5))
    order = np.argsort(-s[pk[:, 0], pk[:, 1]])
    pk, sc = pk[order], s[pk[order, 0], pk[order, 1]]
    elbow = int(np.argmin(np.diff(sc[:200], 2))) + 1                   # sharpest drop in the sorted scores
    return np.stack([ys[pk[:, 0]], xs[pk[:, 1]]], 1), sc, elbow, s


t0 = time.time()
crop_rng = np.random.default_rng(2)
rows = []
for snr, ((r_img, r_c), (q_img, q_c)) in micro.items():
    exemplars = np.stack([crop_at(r_img, c) for c in r_c[:15]])
    centres = crop_rng.uniform(C, HM - C, (2000, 2))
    d_nearest = np.linalg.norm(centres[:, None] - r_c[None], axis=-1).min(1)   # distance to the nearest true particle
    randoms = np.stack([crop_at(r_img, c) for c in centres[:400]])
    clean = np.stack([crop_at(r_img, c) for c in centres[d_nearest > 22][:400]])
    print(f"SNR {snr}: {100 * (d_nearest[:400] < 16).mean():.0f} % of the random crops contain a particle centre")
    for name, use_scs, others in [("MAE only", False, randoms), ("MAE + SCS, random crops", True, randoms),
                                  ("MAE + SCS, clean (oracle)", True, clean)]:
        m = train_mae(exemplars, others, use_scs)
        coords, sc, elbow, smap = pick_with_exemplars(m, q_img, exemplars)
        f_elbow = match_points(coords[:elbow], q_c, 10)["f1"]
        f_count = match_points(coords[:len(q_c)], q_c, 10)["f1"]
        rows.append((snr, name, elbow, f_elbow, f_count))
        if snr == 0.05 and "oracle" in name:
            demo = (q_img, q_c, coords[:elbow], smap)
print(f"(6 trainings and pickings in {time.time() - t0:.0f} s)")
print(f"{'SNR':>5s}  {'model':26s} {'picked':>6s} {'F1 (elbow)':>10s} {'F1 (true count)':>15s}   [true particles: {len(qry_c)}]")
for snr, name, k, fe, fc in rows:
    print(f"{snr:5.2f}  {name:26s} {k:6d} {fe:10.2f} {fc:15.2f}")

In [ ]:
# ── 3.4 Picks on the hardest micrograph ──
q_img, q_c, picks, smap = demo
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.4))
axes[0].imshow(smap, cmap="magma"); axes[0].set_title("max cosine similarity to exemplars (SNR 0.05, oracle)", fontsize=9)
axes[1].imshow(q_img); axes[1].set_title("picks (orange) vs true particles (cyan)")
axes[1].scatter(q_c[:, 1], q_c[:, 0], s=160, facecolors="none", edgecolors="tab:cyan")
axes[1].scatter(picks[:, 1], picks[:, 0], s=40, marker="x", color="tab:orange")
for a in axes:
    a.axis("off")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Read the table row by row (exact numbers vary with the random seed).

* At **SNR 0.1** the masked autoencoder's features alone, compared with 15 exemplars, find a good share of the
  particles (F1 ≈ 0.6 when we keep the true number of picks), although the elbow overshoots and picks about twice as
  many. Self-supervision without any labels already learned much of "what a particle looks like".
* At **SNR 0.05** those features are no longer enough: junk blobs and noise score as high as particles.
* Adding the similarity loss with **random crops** as the "others" does *not* fix this, and at SNR 0.1 it does real
  damage. The printout says why: nearly half of the random crops contain a particle. The loss is told to push those
  particles *away* from the exemplars — **the same "unlabelled ≠ negative" problem as Section 2**, now inside a
  contrastive loss.
* With **clean** others (the oracle control), the similarity loss clearly helps at SNR 0.05 (F1 roughly doubles). At
  SNR 0.1 it adds little to MAE alone: with our guessed weights (α, β, τ) and only 15 exemplars, there is not much left
  to gain.

This is why the CryoMAE paper adjusts its similarity loss for positive–unlabelled data using an estimated positive
fraction — a principled way to get closer to the oracle without knowing the truth. The elbow is a practical way to choose
*how many* to pick without a validation set, but it is fragile when scores decline smoothly. In the paper, CryoMAE reached
an average F1 of 0.591 over five EMPIAR datasets (crYOLO 0.502, Topaz 0.394), and F1 grew with the number of exemplars
(1/5/15/25 exemplars: 0.207/0.373/0.591/0.594).

✏️ **Your turn.** (a) Use only 5 exemplars (`r_c[:5]`) — does F1 drop as in the paper's exemplar ablation?
(b) Set `beta=0.1` in `train_mae` — does a gentler similarity loss stop the damage at SNR 0.1?

---
## 4 · Meta-learning: learning to adapt quickly (illustration)

A picker trained on one microscope, sample and defocus often fails on another. **Meta-learning** trains not a single
good picker but a good **starting point** — weights that can be fine-tuned to a *new* condition with very few labels.
**Reptile** (Nichol *et al.* 2018) is one of the simplest algorithms: repeatedly (1) sample a task, (2) fine-tune a copy of
the weights on it for a few steps, (3) move the shared weights a little toward the fine-tuned ones,

$$\theta \leftarrow \theta + \epsilon\,\big(\tilde\theta_{\text{task}}-\theta\big).$$

⚠️ **RobPicker** ("A meta learning framework for robust identification of macromolecules in cryo-electron tomograms",
Hosseini *et al.*, Nature Methods) uses meta-learning for robust picking, but we could not access its text. The toy below
illustrates meta-learning in general; it is **not** RobPicker's algorithm.

Our tasks: images of disk-like particles whose **size, contrast sign and noise** vary from task to task, plus filament
distractors. A small CNN predicts a heat map of particle centres. We compare three starting points, each fine-tuned for
20 steps on a small, fully annotated 48×48 corner of a new task: random weights, weights pre-trained on all tasks pooled
together, and Reptile weights.

In [ ]:
# ── 4.1 A family of picking tasks ──
S4 = 96


def task_image(rng, radius, contrast, snr, n=14):
    """Disks (with a dark halo) of one size/contrast + filament distractors + noise; returns image, centre heat map, centres."""
    yy_, xx_ = np.mgrid[:S4, :S4]
    img, heat, cs = np.zeros((S4, S4)), np.zeros((S4, S4)), []
    for _ in range(n):
        for _ in range(100):
            c = rng.uniform(radius + 2, S4 - radius - 2, 2)
            if all(np.linalg.norm(c - o) > 2 * radius + 2 for o in cs):
                cs.append(c)
                d2 = (yy_ - c[0]) ** 2 + (xx_ - c[1]) ** 2
                img += contrast * (d2 <= radius**2) - 0.5 * contrast * ((d2 > radius**2) & (d2 <= (radius + 2) ** 2))
                heat = np.maximum(heat, np.exp(-d2 / (2 * 2.0**2)))
                break
    for _ in range(3):
        a = rng.uniform(0, np.pi); y0, x0 = rng.uniform(0, S4, 2)
        img += 0.8 * contrast * np.exp(-(((yy_ - y0) * np.cos(a) - (xx_ - x0) * np.sin(a)) ** 2) / (2 * 1.5**2))
    img = ndimage.gaussian_filter(img, 1.0)
    img = img + rng.normal(0, np.sqrt(img.var() / snr), img.shape)
    return normalize(img), heat.astype(np.float32), np.array(cs)


def random_task(rng):
    return dict(radius=rng.uniform(3, 7), contrast=float(rng.choice([-1.0, 1.0])), snr=rng.uniform(0.3, 1.0))


def picker_net():
    return nn.Sequential(nn.Conv2d(1, 16, 5, padding=2), nn.ReLU(), nn.Conv2d(16, 16, 5, padding=4, dilation=2), nn.ReLU(),
                         nn.Conv2d(16, 1, 5, padding=2))


def heat_loss(net, x, h):
    return F.mse_loss(net(torch.tensor(x)[None, None])[0, 0], torch.tensor(h))


def finetune(net, x, h, steps, lr=1e-2):
    opt = torch.optim.SGD(net.parameters(), lr=lr, momentum=0.9)
    for _ in range(steps):
        loss = heat_loss(net, x, h); opt.zero_grad(); loss.backward(); opt.step()
    return net


def f1_on(net, x, cs, r):
    with torch.no_grad():
        s = net(torch.tensor(x)[None, None])[0, 0].numpy()
    peaks = np.argwhere((s == ndimage.maximum_filter(s, size=7)) & (s > 0.3))
    return match_points(peaks, cs, max(3, r))["f1"]


task_rng = np.random.default_rng(3)
examples = [task_image(task_rng, **random_task(task_rng)) for _ in range(4)]
fig, axes = plt.subplots(1, 4, figsize=(9, 2.4))
for ax, (im, _, _) in zip(axes, examples):
    ax.imshow(im); ax.axis("off")
plt.suptitle("four tasks: particles differ in size, contrast and noise", fontsize=9); plt.show()

In [ ]:
# ── 4.2 Meta-training (Reptile) vs pooled pre-training, then few-shot adaptation to new tasks ──
t0 = time.time()
torch.manual_seed(SEED); meta = picker_net()
torch.manual_seed(SEED); pooled = picker_net(); pooled_opt = torch.optim.Adam(pooled.parameters(), lr=2e-3)
EPS = 0.3
for it in range(300):
    x, h, _ = task_image(task_rng, **random_task(task_rng))
    fast = finetune(copy.deepcopy(meta), x, h, steps=5)                 # inner loop on one task
    with torch.no_grad():
        for p, q in zip(meta.parameters(), fast.parameters()):
            p += EPS * (q - p)                                          # Reptile outer update
    loss = heat_loss(pooled, x, h); pooled_opt.zero_grad(); loss.backward(); pooled_opt.step()
print(f"meta-training and pooled pre-training: {time.time() - t0:.0f} s")

scores4 = {"random init": [], "pooled pre-training": [], "Reptile (meta-learned)": []}
for k in range(8):
    tp = random_task(task_rng)
    x_sup, h_sup, _ = task_image(task_rng, **tp)                        # a new task: annotate one 48×48 corner…
    x_qry, _, c_qry = task_image(task_rng, **tp)                        # …and pick on a fresh image of that task
    for name, init in [("random init", None), ("pooled pre-training", pooled), ("Reptile (meta-learned)", meta)]:
        torch.manual_seed(k)
        net = picker_net() if init is None else copy.deepcopy(init)
        finetune(net, x_sup[:48, :48], h_sup[:48, :48], steps=20)
        scores4[name].append(f1_on(net, x_qry, c_qry, tp["radius"]))
for name, v in scores4.items():
    print(f"{name:24s} F1 on 8 new tasks: {np.round(v, 2)}  mean {np.mean(v):.2f}")

🔍 **What just happened?** Pooled pre-training learns an "average" picker; when a new task flips the contrast it can
fail completely and 20 steps are not enough to recover. Starting from random weights adapts, but slowly. The Reptile
initialisation was *trained to be easy to adapt*, so the same 20 steps on a tiny annotated region go further. Robustness
across microscopes, specimens and acquisition settings is exactly what a meta-learned picker aims for — read the
RobPicker paper for how the lab does it in 3D.

---
## 5 · The supervision ladder of the lab's pickers

| Method (module) | What the user provides | How it learns | Data type |
|---|---|---|---|
| Difference of Gaussians (Pei *et al.* 2016; Module 11) | particle size | no learning | cryo-ET |
| Template matching (Modules 11, 12) | a structure (template) | no learning | cryo-ET |
| Template-free pattern mining, DISCA (Modules 15, 22) | nothing | unsupervised | cryo-ET subtomograms |
| Unsupervised multi-scale segmentation with Stable Diffusion (Module 32) | a few slabs | unsupervised → pseudo-labels | cryo-ET |
| CryoSAM (Module 32) | **one** click | training-free (foundation models) | cryo-ET |
| **TomoPicker** (this module) | ~10 clicks + expected count | positive–unlabelled learning | cryo-ET |
| **CryoMAE** (this module) | ~15 exemplars | self-supervised MAE + similarity | cryo-EM micrographs |
| **RobPicker** (this module) | ⚠️ not verified | meta-learning | cryo-ET |
| Deep reinforcement learning (Cai *et al.* 2019) | ⚠️ details not verified | an agent learns where to look | cryo-ET |
| Faster R-CNN for mitochondria (Li *et al.* 2019; Module 17) | many boxes | supervised detection | cryo-ET 2D slices |
| Supervised 3D CNNs (Module 16) | many labels | supervised | cryo-ET subtomograms |

Moving up the ladder trades annotation effort for assumptions: about the expected number of particles (PU learning),
about what exemplars represent (CryoMAE), or about what generalises across tasks (meta-learning).

---
### 📄 Paper connection — TomoPicker (Uddin *et al.*, Briefings in Bioinformatics 2025)

**Uddin MR, Ahmed AY, Tabib HMS, Tahmid MT, Alam MZU, Freyberg Z, Xu M.** *Localization of macromolecules in crowded
cellular cryo-electron tomograms from extremely sparse labels.* Briefings in Bioinformatics 26(6):bbaf630 (2025).
[doi:10.1093/bib/bbaf630](https://doi.org/10.1093/bib/bbaf630).
- **Question:** find all ribosomes in crowded cellular tomograms when only ~10 have been annotated.
- **What the paper does:** voxel classification with a 3D residual U-Net, trained with **positive–unlabelled** objectives:
  nnPU and GE-KL (λ = 10), with the prior $\pi'=(E_p\times R_n)/|\mathcal P\cup\mathcal U|$ computed from the expected
  particle count; falls back to ordinary PN learning when the observed positive fraction already reaches the expected one;
  keeps a fixed number of top-scoring peaks after non-maximum suppression (2,000 in the released code's example).
- **What our toy keeps:** the PU objectives as written in the released code, π from an expected count, NMS picking.
- **What it leaves out:** the 3D residual U-Net, tiling of whole tomograms into subtomograms, and real data.
- **Data and results reported:** *S. pombe* (EMPIAR-10988, Volta phase plate and defocus-only) and *M. pneumoniae*
  (EMPIAR-10499). With 10 labels on the VPP data, TomoPicker (GE-KL) reached F1 0.46 versus 0.17 for DeepETPicker; with
  500 labels DeepETPicker reached 0.57 — so a gap remains, but at up to 98 % fewer annotations.
- **Code:** [AITom `pick/ml/TomoPicker`](https://github.com/xulabs/aitom/tree/master/aitom/pick/ml/TomoPicker).

### 📄 Paper connection — CryoMAE (Xu, Zhan & Xu, WACV 2025)

**Xu C, Zhan X, Xu M.** *CryoMAE: Few-shot cryo-EM particle picking with masked autoencoders.* WACV 2025, pp. 3876–3885.
[doi:10.1109/WACV61041.2025.00381](https://doi.org/10.1109/WACV61041.2025.00381); [arXiv:2404.10178](https://arxiv.org/abs/2404.10178).
- **What the paper does:** a ViT masked autoencoder (224×224 inputs, 16×16 patches, 75 % masking) trained on ~15
  exemplars and unlabelled crops with an added self–cross similarity loss (PU-adjusted with an estimated positive
  fraction); sliding-window max-cosine scoring (stride 28 px); per-micrograph elbow threshold on k-nearest-neighbour
  distances.
- **Results reported:** average F1 0.591 vs 0.502 (crYOLO) and 0.394 (Topaz) on CryoPPP subsets of five EMPIAR datasets;
  average 3D resolution 10.13 Å vs 11.40 Å (Topaz) and 12.10 Å (crYOLO) — the abstract's "up to 22.4 %" improvement is
  the best case, the average is about 11 %.
- **Our toy keeps / leaves out:** keeps masked reconstruction, the SCS loss form, exemplar max-cosine scoring and an elbow
  cut; leaves out the ViT, the PU adjustment (our oracle control shows why it is needed), the exact threshold rule and
  real micrographs.

### 📄 Paper connection — RobPicker (Hosseini *et al.*, Nature Methods)

⚠️ **Content not verified.** Listed on the lab's CV as *RobPicker: A meta learning framework for robust identification of
macromolecules in cryo-electron tomograms*, Nature Methods. We could not locate its text, year or DOI while writing this
module. Section 4 illustrates meta-learning in general — read the paper for its actual formulation and results.

### 📄 Paper connection — deep reinforcement learning for localisation (Cai *et al.*, NeurIPS DRL workshop 2019)

**Cai Y, Zeng X, Zeng Y, Liu W, Jin J, Freyberg Z, Yang G, Xu M.** *3D macromolecule localization in cryo-electron
tomography with deep reinforcement learning.* NeurIPS Deep Reinforcement Learning Workshop 2019.
⚠️ Only the citation could be verified. From the title: an agent learns a policy for moving through a tomogram toward
macromolecules, instead of scoring every voxel.

## ✅ Checkpoint

1. Why is "unlabelled" not the same as "negative" in particle picking?
   <details><summary>Answer</summary>The unlabelled region contains all the particles nobody clicked; calling them negative teaches the model to reject real particles.</details>
2. What does a PN classifier trained on labelled-vs-unlabelled data get right, and what does it get wrong?
   <details><summary>Answer</summary>Under random labelling its ranking of pixels can be correct, but its probabilities are shrunk by the labelling fraction c, so fixed thresholds fail (and deep networks may also memorise the labelled instances).</details>
3. How does TomoPicker obtain π without knowing the true labels?
   <details><summary>Answer</summary>From an expected particle count E_p and the particle volume R_n: π' = E_p·R_n / (number of voxels).</details>
4. What does the max(0, ·) in nnPU prevent?
   <details><summary>Answer</summary>The estimated negative risk becoming negative (impossible for a true risk), which would otherwise let a flexible model overfit.</details>
5. What does the self–cross similarity loss encourage?
   <details><summary>Answer</summary>Exemplar embeddings close to each other (high S_self) and far from random crops (low S_cross).</details>
6. What does meta-learning optimise, compared with ordinary pre-training?
   <details><summary>Answer</summary>Ordinary pre-training optimises performance on the pooled training data; meta-learning optimises how well the weights perform *after a few steps of adaptation* to a new task.</details>

## 🧠 Questions & discussion

1. Clicked particles are rarely a random sample — experts click the clearest ones. How does that violate the PU
   assumptions, and what might it do to a picker?
2. TomoPicker needs an expected particle count. How would you estimate it for a new cell type?
3. CryoMAE's exemplars come from one micrograph. What could go wrong if that micrograph has a different defocus or ice
   thickness than the others?
4. Which is cheaper for a lab: 10 clicks per tomogram for 100 tomograms, or 500 clicks once? What else matters besides
   clicks?
5. How would you combine CryoSAM's one-click segmentation (Module 32) with PU learning?

## 🚀 Bonus — going further

- **Deep PU.** Replace the logistic model of Section 2 with a small fully convolutional network trained on crops. Does
  PN's *ranking* suffer once the model can memorise the clicked particles?
- **Real data.** Run AITom's TomoPicker (`train_tomopicker.py`) on an EMPIAR-10988 tomogram with 10 labels.
- **Better elbows.** Implement the paper's k-nearest-neighbour-distance elbow and compare it with ours.
- **MAML.** Implement first-order MAML instead of Reptile and compare adaptation curves.

## 📝 Lab note

```text
Module 31 — date:
Input: crowded 2D slabs with 10 labelled ribosomes; β-galactosidase micrographs with 15 exemplars; a family of picking tasks
Output: probability maps, F1 (threshold and top-k), exemplar-based picks, few-shot adaptation scores
Method: PN vs nnPU vs GE-KL; TinyMAE ± self–cross similarity; Reptile vs pooled pre-training
Parameter I changed and what happened:
A failure mode / surprise:
Paper link: Uddin et al. 2025 (TomoPicker); Xu, Zhan & Xu 2025 (CryoMAE); RobPicker (unverified)
```

## 📚 References

1. Uddin MR *et al.* Localization of macromolecules in crowded cellular cryo-electron tomograms from extremely sparse labels. *Brief Bioinform* 26:bbaf630 (2025). doi:10.1093/bib/bbaf630
2. Xu C, Zhan X, Xu M. CryoMAE: Few-shot cryo-EM particle picking with masked autoencoders. *WACV* 2025, pp. 3876–3885. doi:10.1109/WACV61041.2025.00381
3. Hosseini *et al.* RobPicker: A meta learning framework for robust identification of macromolecules in cryo-electron tomograms. *Nature Methods* (details not verified).
4. Cai Y *et al.* 3D macromolecule localization in cryo-electron tomography with deep reinforcement learning. NeurIPS Deep RL Workshop 2019.
5. Kiryo R, Niu G, du Plessis MC, Sugiyama M. Positive-unlabeled learning with non-negative risk estimator. *NeurIPS* 2017.
6. Elkan C, Noto K. Learning classifiers from only positive and unlabeled data. *KDD* 2008.
7. He K *et al.* Masked autoencoders are scalable vision learners. *CVPR* 2022.
8. Nichol A, Achiam J, Schulman J. On first-order meta-learning algorithms. arXiv:1803.02999 (2018).

## ➡️ Next

**[32 · Segmenting the cellular world: foundation models, diffusion features and topology](32_segmenting_the_cellular_world.ipynb)** —
from points to masks: segment membranes, organelles and molecules with one click, no labels, and correct topology.
Course overview: [`README.md`](../README.md) · terms: [`GLOSSARY.md`](../GLOSSARY.md).